전체 논의 내용을 바탕으로 데이터 전처리부터 모델링, 평가, 핵심 개념까지 초보자도 쉽게 이해할 수 있도록 구성한 주피터 노트북 교재입니다. 주피터 노트북 파일(`.ipynb`)의 마크다운 셀과 코드 셀에 그대로 복사해서 사용하실 수 있도록 작성했습니다.

---

# [마크다운 셀 1]

# 머신러닝 기초 및 Ace Associate 대비 완전정복 가이드북

이 교재는 데이터 전처리부터 모델 평가, 그리고 머신러닝의 필수 핵심 개념(불리언 인덱싱, 결측치 처리, 원핫 인덱싱, 데이터 누수 방지, 하이퍼파라미터, 모델 평가 지표)을 초보자 눈높이에 맞춰 다룹니다.

---

## 1. 불리언 인덱싱(Boolean Indexing)으로 데이터 필터링하기

### 개념 이해

* **원리**: 조건식을 실행하면 데이터의 각 행마다 `True` 또는 `False`로 구성된 스위치 목록(Mask)이 만들어집니다.
* **필터링**: 데이터프레임 대괄호 `df[...]` 안에 이 스위치 목록을 넣으면 **`True`에 해당하는 행(Row)만 추출**됩니다. (열이 아닌 행이 남습니다)

In [ ]:
# [코드 셀 1]
import pandas as pd
import numpy as np

# 연습용 샘플 데이터프레임 생성
data = {
    'Distance': [100, 250, 400, 150],
    'Speed_Per_Hour': [80, 320, 250, 350],
    'Address1': ['경기도평택시', '경기도의정부시', '경기도광명시', '경기도평택시'],
    'Address2': ['A동', 'B동', 'A동', 'C동'],
    'DrivingTime': [75, 50, 100, 30] # 주행시간 (분)
}
df = pd.DataFrame(data)

print("--- 원본 데이터 ---")
print(df)

# 속도가 300 미만인 데이터만 필터링
df_temp = df[df.Speed_Per_Hour < 300]

print("\n--- Speed_Per_Hour < 300 필터링 결과 ---")
print(df_temp)

---

# [마크다운 셀 2]

## 2. 결측치(Null/NaN) 확인 및 제거

### `isnull().sum()`의 원리

1. `df.isnull()`: 원본과 똑같은 크기의 표를 만들고, 값이 비어있으면 `True`, 채워져 있으면 `False`를 반환합니다.
2. `.sum()`: 파이썬은 `True`를 1, `False`를 0으로 계산합니다. 위에서 아래(세로/열 방향)로 덧셈을 수행하여 **각 열별 결측치 개수**를 구합니다.

### `dropna()`의 원리

* 결측치가 하나라도 포함된 행(Row)을 통째로 삭제합니다. (열을 삭제하려면 `axis=1` 옵션 필요)

In [ ]:
# [코드 셀 2]
# 강제로 결측치(NaN) 삽입
df_na = df_temp.copy()
df_na.loc[0, 'Distance'] = np.nan

print("--- 결측치 확인 (isnull().sum()) ---")
print(df_na.isnull().sum())

# 결측치가 포함된 행 제거
df_clean = df_na.dropna()

print("\n--- dropna() 적용 후 데이터 ---")
print(df_clean)

---

# [마크다운 셀 3]

## 3. 원핫 인덱싱(One-Hot Encoding)과 열 증가 원리

### `pd.get_dummies()` 동작 방식

* **범주형(문자열) 데이터**를 머신러닝 모델이 이해할 수 있는 숫자 형태(0과 1)로 변환합니다.
* **열 개수**: $2^n$이 아니라 **범주(Class)의 고유 개수만큼** 생성됩니다. (예: 범주가 3개면 열 3개 추가)
* **One-Hot 표기**: 생성된 열 중에서 해당되는 단 하나의 위치만 `1`(Hot)이 켜지고, 나머지는 모두 `0`이 됩니다. (`100`, `010`, `001` 형태)

In [ ]:
# [코드 셀 3]
# Address1(3가지 종류)과 Address2(3가지 종류) 원핫 인코딩
df_preset = pd.get_dummies(data=df_clean, columns=['Address1', 'Address2'])

print("--- 원핫 인코딩 적용 후 데이터프레임 ---")
print(df_preset)

---

# [마크다운 셀 4]

## 4. 데이터 분할 및 `random_state`의 정확한 의미

### `random_state`란?

* 컴퓨터가 난수를 생성할 때 사용하는 **시드(Seed) 번호**입니다.
* **설정 이유**: `random_state=42`처럼 정수를 지정해 두면 몇 번을 다시 실행해도 **항상 똑같은 조합으로 데이터가 분할**되어 코드의 재현 가능성(Reproducibility)과 공정한 모델 비교 환경을 보장합니다.

In [ ]:
# [코드 셀 4]
from sklearn.model_selection import train_test_split

# 독립변수(X)와 종속변수(y) 분리
X = df_preset.drop('DrivingTime', axis=1)
y = df_preset['DrivingTime']

# Train(80%) / Valid(20%) 분할
X_train, X_valid, y_train, y_valid = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"X_train 크기: {X_train.shape}, X_valid 크기: {X_valid.shape}")

---

# [마크다운 셀 5]

## 5. RobustScaler 및 데이터 누수(Data Leakage) 방지

### RobustScaler 특징

* 중앙값(Median)과 사분위수 범위(IQR)를 사용하므로 **이상치(Outlier)의 영향에 강한** 스케일러입니다.

### 핵심 개념: 데이터 누수(Data Leakage)란?

* **원인**: `fit()`을 Train 데이터뿐만 아니라 Valid/Test 데이터에 적용하거나 두 데이터를 합쳐서 수행할 때 발생합니다.
* **문제점**: 검증지(Valid)의 통계 정보(힌트)가 모델/스케일러에 유출되어 모의고사 점수만 높아지고, **실전에서의 일반화 성능(Generalization Performance)을 상실**하게 됩니다.
* **올바른 규칙**: Train 데이터에는 `.fit_transform()`을 사용하고, Valid/Test 데이터에는 오직 Train의 기준값으로 **`.transform()`만 적용**해야 합니다.

In [ ]:
# [코드 셀 5]
from sklearn.preprocessing import RobustScaler

rs = RobustScaler()

# Train 기준값 학습 및 변환 (fit_transform)
X_train_scaled = rs.fit_transform(X_train)

# Valid는 Train 기준을 그대로 적용 (transform만 실행 - 데이터 누수 방지)
X_valid_scaled = rs.transform(X_valid)

# 스케일링된 Valid 데이터의 최댓값 정수 반올림 예시
max_val = round(np.max(X_valid_scaled))
print(f"X_valid 최댓값 반올림: {max_val}")

---

# [마크다운 셀 6]

## 6. DecisionTree vs RandomForest 모델링 및 하이퍼파라미터

### 핵심 하이퍼파라미터

* `max_depth=5`: 트리의 최대 질문 단계를 5단계로 제한하여 복잡도 감소 (과대적합 방지)
* `min_samples_split=3`: 노드에 최소 3개 이상의 데이터가 남아있어야만 추가 분할 수행
* `random_state=120`: 트리의 무작위 분할 과정을 고정하여 결과 재현

### 모델 차이점

* **DecisionTree**: 1명의 전문가(나무 1개)가 판단
* **RandomForest**: 여러 나무(기본 100개)의 예측 결과를 평균 내는 **앙상블(집단지성)** 모델

In [ ]:
# [코드 셀 6]
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor

# 모델 선언
dt = DecisionTreeRegressor(max_depth=5, min_samples_split=3, random_state=120)
rf = RandomForestRegressor(max_depth=5, min_samples_split=3, random_state=120)

# 모델 학습 (Train 데이터 이용)
dt.fit(X_train_scaled, y_train)
rf.fit(X_train_scaled, y_train)

---

# [마크다운 셀 7]

## 7. Feature Importance(특성 중요도) 해석

### 해석법

* 각 변수가 주행시간(DrivingTime)을 예측하는 데 얼마나 기여했는지 보여줍니다.
* 물리학의 거속시 공식($시간 = 거리 \div 속도$)에 따라 **Distance와 Speed_Per_Hour 변수의 중요도가 전체의 대다수를 차지**하게 되며, 이는 모델이 상식에 부합하게 데이터의 규칙을 제대로 학습했음을 의미합니다.

In [ ]:
# [코드 셀 7]
# Random Forest의 특성 중요도 확인
feature_importances = pd.DataFrame({
    'Feature': X.columns,
    'Importance': rf.feature_importances_
}).sort_values(by='Importance', ascending=False)

print("--- 특성 중요도(Feature Importance) 결과 ---")
print(feature_importances)

---

# [마크다운 셀 8]

## 8. 모델 평가 지표: MAE(Mean Absolute Error) 해석

### MAE 계산법

* **공식**: $\vert{}실제값 - 예측값\vert{}$의 평균 (루트 적용 안 함)
* **특징**: 오차가 커질수록 성능이 좋지 않으며, **숫자가 작을수록 예측이 정밀한 우수한 모델**입니다.
* **단위**: 원본 타깃 변수의 단위(분)를 그대로 따릅니다.
* 예: MAE가 59.16이면 예측값과 실제값 사이에 **평균 59.16분의 오차가 존재**한다는 뜻입니다.


* **결과**: 단일 나무(DecisionTree)보다 집단지성을 활용하는 숲(RandomForest)이 오차를 크게 줄여 월등한 성능을 보입니다.

In [ ]:
# [코드 셀 8]
from sklearn.metrics import mean_absolute_error

# Valid 데이터 예측
y_pred_dt = dt.predict(X_valid_scaled)
y_pred_rf = rf.predict(X_valid_scaled)

# MAE 계산
dt_mae = mean_absolute_error(y_valid, y_pred_dt)
rf_mae = mean_absolute_error(y_valid, y_pred_rf)

print(f"DecisionTree MAE: {dt_mae:.2f} (평균 오차 {dt_mae:.2f}분)")
print(f"RandomForest MAE: {rf_mae:.2f} (평균 오차 {rf_mae:.2f}분)")